# Condition Structure Discovery

The existing notebooks pin the analysis to two *supervised / representative* lenses:
`oracle_signal_test` → the **K representatives**; `condition_informativeness` → the
**4 query types**. Both assume what the structure is, then measure against it.

This notebook is **unsupervised-first**: it looks for structure in the full
`[N, 16]` per-sample condition field with no rep/type prior, then grounds it.

| Part | Question | Method |
|---|---|---|
| A | Is the condition manifold genuinely structured (vs a blob)? | Intrinsic dim, unsupervised clustering, Gaussian null |
| B | Did training **preserve the buddy graph** it was seeded from? | kNN-overlap & drift: trained vs init vs `E = A_img ∪ A_txt` |
| C | What does the discovered structure **mean**? | Probe conditions from img/txt/type/gap; profile clusters & axes |

All conditions (init / trained / type) are row-aligned by the same `sample_ids`.

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
EXPERIMENT_DIR = (
    "/project/CoSiR/res/CoSiR_Experiment_buddy/impressions/20260609_184032_CoSiR_Experiment"
)
BASE_DIR      = "/project/CoSiR/res/CoSiR_Experiment_buddy/impressions"  # holds template_embeddings (buddy init)
FEATURE_STORE = "/data/SSD2/pre_extract/impressions/features"

DEVICE     = "cuda"
KNN_K      = 15      # neighborhood size for graph-overlap metrics
BUDDY_K    = 30      # K for rebuilding the buddy graph (matches train.buddies.k)
N_SUB      = 2500    # subsample for O(N^2) metrics (trustworthiness)
SEED       = 42
TYPE_NAMES  = ["caption", "description", "impression", "aesthetic"]
TYPE_COLORS = ["#888888", "#2196F3", "#FF9800", "#4CAF50"]

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, glob, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, adjusted_mutual_info_score,
                             adjusted_rand_score)
from sklearn.manifold import trustworthiness
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
warnings.filterwarnings("ignore")

import sys
_proj = "/project/CoSiR"   # repo root (paths below are absolute anyway)
# Force the local repo ahead of any stale site-packages 'src' (which only has main.py)
sys.path.insert(0, _proj)
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    if "site-packages" in getattr(sys.modules[_m], "__file__", "") or "":
        del sys.modules[_m]
from src.utils import FeatureManager
from src.conditional_buddy import build_buddy_graphs

rng = np.random.default_rng(SEED)
np.random.seed(SEED); torch.manual_seed(SEED)
print("Imports OK")

In [ ]:
# ── Load conditions, types, features — all aligned to one canonical order ────
snap_files = sorted(glob.glob(os.path.join(EXPERIMENT_DIR, "condition_viz", "epoch_*.pt")))
snap = torch.load(snap_files[-1], map_location="cpu", weights_only=False)
cond_trained = snap["label_embeddings_all"].float().numpy()   # [N, 16]  (S_cond order)
types        = snap["train_sample_types"].numpy()             # [N]      (S_cond order)
reps         = snap["representatives"].float().numpy()        # [K, 16]
combine_side = snap["combine_side"]
N, D = cond_trained.shape
print(f"Trained conditions: {cond_trained.shape} | combine_side={combine_side} | K_reps={len(reps)}")

# Buddy init (template_embeddings) — same sample_ids ordering as final/snapshot
cond_init = np.load(os.path.join(BASE_DIR, "template_embeddings", "embeddings.npy")).astype(np.float32)
init_ids  = np.load(os.path.join(BASE_DIR, "template_embeddings", "sample_ids.npy"))
fin_ids   = np.load(os.path.join(EXPERIMENT_DIR, "final_embeddings", "sample_ids.npy"))
assert np.array_equal(init_ids, fin_ids), "init/trained sample_id ordering mismatch"
sample_ids = fin_ids
# sanity: snapshot row order matches final_embeddings row order
fin_emb = np.load(os.path.join(EXPERIMENT_DIR, "final_embeddings", "embeddings.npy")).astype(np.float32)
assert np.abs(fin_emb - cond_trained).mean() < 1e-2, "snapshot vs final_embeddings order mismatch"

# Features — reorder from store order to S_cond order
fm = FeatureManager(FEATURE_STORE)
feat_ids = np.asarray(fm.get_all_sample_ids())
fdata = fm.load_all_to_ram(["img_features", "txt_features"])
img_all = fdata["img_features"].numpy().astype(np.float32)
txt_all = fdata["txt_features"].numpy().astype(np.float32)
pos = {int(s): i for i, s in enumerate(feat_ids)}
sel = np.array([pos[int(s)] for s in sample_ids])
img_feat = img_all[sel]; txt_feat = txt_all[sel]
gap_mag  = np.linalg.norm(img_feat / (np.linalg.norm(img_feat,axis=1,keepdims=True)+1e-8)
                          - txt_feat / (np.linalg.norm(txt_feat,axis=1,keepdims=True)+1e-8), axis=1)

print(f"init {cond_init.shape}  trained {cond_trained.shape}  img {img_feat.shape}  txt {txt_feat.shape}")
print(f"type counts: {dict(zip(TYPE_NAMES, np.bincount(types).tolist()))}")

---
## Part A — Geometric: is there structure at all?

No type/rep prior. Three views: how many dimensions the field really uses,
how many clusters fall out unsupervised, and whether that beats a Gaussian null.

In [ ]:
# ── A1: Intrinsic dimensionality (init vs trained) ───────────────────────────
def participation_ratio(X):
    ev = np.linalg.svd(X - X.mean(0), compute_uv=False) ** 2
    return float(ev.sum() ** 2 / (ev ** 2).sum())

pca_i = PCA().fit(cond_init); pca_t = PCA().fit(cond_trained)
PR_i, PR_t = participation_ratio(cond_init), participation_ratio(cond_trained)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.plot(np.arange(1, D+1), np.cumsum(pca_i.explained_variance_ratio_), "o-", color="#888", label=f"init (PR={PR_i:.2f})")
ax.plot(np.arange(1, D+1), np.cumsum(pca_t.explained_variance_ratio_), "s-", color="#2196F3", label=f"trained (PR={PR_t:.2f})")
ax.axhline(0.9, color="r", ls="--", lw=1); ax.set_xlabel("# PCA components"); ax.set_ylabel("cum. explained var")
ax.set_title("Effective dimensionality"); ax.legend(); ax.spines[["top","right"]].set_visible(False)

ax2 = axes[1]
w = 0.4
ax2.bar(np.arange(D)-w/2, pca_i.explained_variance_ratio_, w, color="#888", label="init")
ax2.bar(np.arange(D)+w/2, pca_t.explained_variance_ratio_, w, color="#2196F3", label="trained")
ax2.set_xlabel("PCA component"); ax2.set_ylabel("explained var ratio"); ax2.set_title("Per-component variance")
ax2.legend(); ax2.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"Participation ratio  init={PR_i:.2f}  trained={PR_t:.2f}  (of {D} dims)")
print("→ PR≈D: field uses all dims; PR≪D: training collapsed conditions toward a low-dim subspace.")

In [ ]:
# ── A2: Unsupervised clustering — how many groups, and do they match the 4 types? ──
mcs = max(50, N // 200)
hdb = HDBSCAN(min_cluster_size=mcs)
lab = hdb.fit_predict(cond_trained)
mask = lab != -1
n_clusters = len(set(lab[mask]))
noise_frac = float((~mask).mean())

ks = list(range(2, 16))
km_labels = {k: KMeans(k, n_init=10, random_state=SEED).fit_predict(cond_trained) for k in ks}
sils = [silhouette_score(cond_trained, km_labels[k], sample_size=min(3000, N), random_state=SEED) for k in ks]
best_k = ks[int(np.argmax(sils))]

ami = adjusted_mutual_info_score(types[mask], lab[mask]) if n_clusters > 1 else float("nan")
ari = adjusted_rand_score(types[mask], lab[mask]) if n_clusters > 1 else float("nan")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
ax = axes[0]
ax.plot(ks, sils, "o-", color="#2196F3"); ax.axvline(4, color="r", ls="--", lw=1, label="#query types=4")
ax.axvline(best_k, color="#4CAF50", ls=":", lw=1.5, label=f"best k={best_k}")
ax.set_xlabel("k (KMeans)"); ax.set_ylabel("silhouette"); ax.set_title("Natural #clusters (KMeans sweep)")
ax.legend(); ax.spines[["top","right"]].set_visible(False)

# contingency: discovered cluster (rows) x query type (cols)
ax2 = axes[1]
if n_clusters > 1:
    uniq = sorted(set(lab[mask]))
    cont = np.zeros((len(uniq), 4))
    for r, c in enumerate(uniq):
        for t in range(4):
            cont[r, t] = np.sum((lab == c) & (types == t))
    cont_n = cont / cont.sum(1, keepdims=True)
    im = ax2.imshow(cont_n, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax2.set_xticks(range(4)); ax2.set_xticklabels([n[:3] for n in TYPE_NAMES])
    ax2.set_yticks(range(len(uniq))); ax2.set_yticklabels([f"C{c}" for c in uniq])
    ax2.set_xlabel("query type"); ax2.set_ylabel("discovered cluster")
    plt.colorbar(im, ax=ax2)
ax2.set_title(f"HDBSCAN clusters vs types\nAMI={ami:.3f} ARI={ari:.3f}")

# PCA scatter colored by discovered cluster
ax3 = axes[2]
pc = PCA(2, random_state=SEED).fit_transform(cond_trained)
sc = ax3.scatter(pc[mask,0], pc[mask,1], c=lab[mask], cmap="tab20", s=4, alpha=0.6)
ax3.scatter(pc[~mask,0], pc[~mask,1], c="lightgray", s=3, alpha=0.3, label="noise")
ax3.set_title(f"Conditions (PCA2)\n{n_clusters} clusters, {noise_frac*100:.0f}% noise")
ax3.set_xlabel("PC1"); ax3.set_ylabel("PC2"); ax3.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"HDBSCAN: {n_clusters} clusters (min_size={mcs}), noise={noise_frac*100:.1f}%")
print(f"Best KMeans k by silhouette: {best_k}  (4 query types as reference)")
print(f"Discovered-vs-type agreement: AMI={ami:.3f}  ARI={ari:.3f}")
print("→ AMI≈1: structure IS the 4 types. AMI≈0: structure is orthogonal to types (the 'bigger picture').")
print(f"→ best_k≠4: condition space is {'finer' if best_k>4 else 'coarser'} than the type lens.")

In [ ]:
# ── A3: Real structure vs Gaussian null ──────────────────────────────────────
cov = np.cov(cond_trained.T); mean = cond_trained.mean(0)
null = rng.multivariate_normal(mean, cov, size=N).astype(np.float32)
lab_n = HDBSCAN(min_cluster_size=mcs).fit_predict(null)
mask_n = lab_n != -1
nclu_n = len(set(lab_n[mask_n]))

sil_real = silhouette_score(cond_trained, km_labels[best_k], sample_size=min(3000, N), random_state=SEED)
sil_null = silhouette_score(null, KMeans(best_k, n_init=10, random_state=SEED).fit_predict(null),
                            sample_size=min(3000, N), random_state=SEED)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["trained\nconditions", "Gaussian\nnull (same cov)"], [sil_real, sil_null],
       color=["#2196F3", "#bdbdbd"], width=0.5)
ax.set_ylabel(f"silhouette (KMeans k={best_k})"); ax.set_title("Structure vs covariance-matched null")
for i, v in enumerate([sil_real, sil_null]): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()

print(f"HDBSCAN clusters: real={n_clusters}  null={nclu_n}")
print(f"Silhouette: real={sil_real:.3f}  null={sil_null:.3f}  (real≫null → genuine clustered structure)")

---
## Part B — Buddy-graph fidelity (the genuinely new question)

The conditions were *initialized* from the buddy graph `E = A_img ∪ A_txt`.
Did training **retain** that structure or wash it out? And did the conditions
drift toward the **image** side or the **text** side?

In [ ]:
# ── B1: Rebuild the buddy graphs from features (same K as training) ──────────
A_img, A_txt, E = build_buddy_graphs(img_feat, txt_feat, K=BUDDY_K, device=DEVICE)
deg_E = np.asarray((E > 0).sum(1)).ravel()
n_comp = connected_components(E, directed=False, return_labels=False)
print(f"E: avg deg {deg_E.mean():.2f}, {n_comp} components, {int((deg_E==0).sum())} isolated")
print(f"A_img avg deg {A_img.nnz/N:.2f} | A_txt avg deg {A_txt.nnz/N:.2f}")

In [ ]:
# ── B2: Neighborhood overlap — does condition-space kNN match the buddy graph? ──
def knn_sets(X, k):
    nn = NearestNeighbors(n_neighbors=k+1).fit(X)
    idx = nn.kneighbors(return_distance=False)[:, 1:]
    return [set(r.tolist()) for r in idx]

def csr_neighbor_sets(A):
    A = A.tocsr()
    return [set(A.indices[A.indptr[i]:A.indptr[i+1]].tolist()) for i in range(A.shape[0])]

def mean_jaccard(SA, SB):
    js = []
    for a, b in zip(SA, SB):
        u = len(a | b)
        if u: js.append(len(a & b) / u)
    return float(np.mean(js)) if js else 0.0

S_tr = knn_sets(cond_trained, KNN_K)
S_in = knn_sets(cond_init, KNN_K)
S_E, S_Ai, S_At = csr_neighbor_sets(E), csr_neighbor_sets(A_img), csr_neighbor_sets(A_txt)

overlaps = {
    "init  vs E":        mean_jaccard(S_in, S_E),
    "trained vs E":      mean_jaccard(S_tr, S_E),
    "trained vs init":   mean_jaccard(S_tr, S_in),
    "trained vs A_img":  mean_jaccard(S_tr, S_Ai),
    "trained vs A_txt":  mean_jaccard(S_tr, S_At),
}
fig, ax = plt.subplots(figsize=(8, 4))
cols = ["#888", "#2196F3", "#9C27B0", "#FF5722", "#4CAF50"]
ax.bar(list(overlaps), list(overlaps.values()), color=cols)
ax.set_ylabel(f"mean kNN Jaccard (k={KNN_K})"); ax.set_title("Condition-space neighborhoods vs buddy graph")
for i, v in enumerate(overlaps.values()): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=20, ha="right"); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

for k, v in overlaps.items(): print(f"  {k:18s}: {v:.3f}")
print("→ 'init vs E' high = init faithfully encodes E. 'trained vs E' relative to it = retention.")
print("→ trained vs A_img  ⪵  trained vs A_txt  reveals which modality training drifted toward.")

In [ ]:
# ── B3: Buddy pairs stay close? + trustworthiness vs each modality ───────────
def buddy_vs_random_ratio(X, A, n_rand=20000):
    A = A.tocoo()
    d_buddy = np.linalg.norm(X[A.row] - X[A.col], axis=1).mean()
    ri = rng.integers(0, len(X), n_rand); rj = rng.integers(0, len(X), n_rand)
    d_rand = np.linalg.norm(X[ri] - X[rj], axis=1).mean()
    return d_buddy / (d_rand + 1e-9)

ratio_init = buddy_vs_random_ratio(cond_init, E)
ratio_train = buddy_vs_random_ratio(cond_trained, E)

# trustworthiness: does condition space preserve feature-space neighborhoods? (subsampled)
sub = rng.choice(N, size=min(N_SUB, N), replace=False)
tw_img = trustworthiness(img_feat[sub], cond_trained[sub], n_neighbors=10)
tw_txt = trustworthiness(txt_feat[sub], cond_trained[sub], n_neighbors=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.bar(["init", "trained"], [ratio_init, ratio_train], color=["#888", "#2196F3"], width=0.5)
ax.axhline(1.0, color="r", ls="--", lw=1, label="random (no structure)")
ax.set_ylabel("buddy / random distance ratio"); ax.set_title("Do E-buddies stay close in condition space?\n(lower = closer = structure kept)")
for i, v in enumerate([ratio_init, ratio_train]): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
ax.legend(); ax.spines[["top","right"]].set_visible(False)

ax2 = axes[1]
ax2.bar(["img features", "txt features"], [tw_img, tw_txt], color=["#FF5722", "#4CAF50"], width=0.5)
ax2.set_ylabel("trustworthiness (k=10)"); ax2.set_title(f"Condition space preserves which modality?\n(subsample N={len(sub)})")
for i, v in enumerate([tw_img, tw_txt]): ax2.text(i, v, f"{v:.3f}", ha="center", va="bottom")
ax2.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"Buddy/random distance ratio: init={ratio_init:.3f}  trained={ratio_train:.3f}")
print(f"Trustworthiness: img={tw_img:.3f}  txt={tw_txt:.3f}  (higher side = condition geometry tracks that modality)")

In [ ]:
# ── B4: Drift — how far, and who moved? ──────────────────────────────────────
disp = np.linalg.norm(cond_trained - cond_init, axis=1)
from scipy.stats import spearmanr
rho_deg, _ = spearmanr(deg_E, disp)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.hist(disp, bins=50, color="#9C27B0", alpha=0.8)
ax.axvline(disp.mean(), color="#FF5722", lw=2, label=f"mean={disp.mean():.3f}")
ax.set_xlabel("‖trained − init‖ per sample"); ax.set_ylabel("# samples")
ax.set_title("Init→trained displacement"); ax.legend(); ax.spines[["top","right"]].set_visible(False)

ax2 = axes[1]
data = [disp[types == t] for t in range(4)]
bp = ax2.boxplot(data, labels=[n[:3] for n in TYPE_NAMES], showfliers=False, patch_artist=True)
for patch, c in zip(bp["boxes"], TYPE_COLORS): patch.set_facecolor(c); patch.set_alpha(0.7)
ax2.set_ylabel("displacement"); ax2.set_title(f"Displacement by query type\n(deg(E) vs disp Spearman ρ={rho_deg:.3f})")
ax2.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"Mean displacement: {disp.mean():.3f}  (init values are in [-1,1] per rank-norm)")
print(f"Hub effect: deg(E) vs displacement ρ={rho_deg:.3f}  (>0: high-degree hubs moved more)")

---
## Part C — Semantic grounding of the discovered structure

Structure you can *predict from content* is real; structure that's per-sample
memorization isn't. Probe how much of the condition vector each factor explains,
then profile the discovered clusters and principal axes.

In [ ]:
# ── C1: Which factors predict the condition vector? (CV R²) ──────────────────
def cv_r2(X, Y):
    return float(cross_val_score(Ridge(alpha=1.0), X, Y, cv=3, scoring="r2").mean())

type_oh = OneHotEncoder(sparse_output=False).fit_transform(types.reshape(-1, 1))
factors = {
    "img features": img_feat,
    "txt features": txt_feat,
    "query type":   type_oh,
    "img−txt gap":  gap_mag.reshape(-1, 1),
}
r2 = {name: cv_r2(X, cond_trained) for name, X in factors.items()}

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(r2), list(r2.values()), color=["#FF5722", "#4CAF50", "#2196F3", "#FF9800"])
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("3-fold CV R² (→ condition vector)")
ax.set_title("How much of the condition does each factor explain?")
for i, v in enumerate(r2.values()): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
plt.xticks(rotation=15); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

for k, v in r2.items(): print(f"  {k:14s} R²={v:.3f}")
print("→ High img/txt R²: conditions are grounded in content (predictable, learnable).")
print("→ R²≈0 everywhere: conditions are per-sample idiosyncratic (no transferable structure).")

In [ ]:
# ── C2: Profile the discovered clusters ──────────────────────────────────────
if n_clusters > 1:
    uniq = sorted(set(lab[mask]))
    rows = []
    for c in uniq:
        m = lab == c
        rows.append([int(m.sum()),
                     *[ (types[m] == t).mean() for t in range(4) ],
                     gap_mag[m].mean()])
    arr = np.array(rows)
    fig, axes = plt.subplots(1, 2, figsize=(13, 0.5*len(uniq)+2.5))
    ax = axes[0]
    im = ax.imshow(arr[:, 1:5], cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels([n[:3] for n in TYPE_NAMES])
    ax.set_yticks(range(len(uniq))); ax.set_yticklabels([f"C{c} (n={int(arr[i,0])})" for i, c in enumerate(uniq)])
    ax.set_title("Cluster × query-type composition")
    for i in range(len(uniq)):
        for j in range(4): ax.text(j, i, f"{arr[i,1+j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax)
    ax2 = axes[1]
    ax2.barh(range(len(uniq)), arr[:, 5], color="#FF9800")
    ax2.set_yticks(range(len(uniq))); ax2.set_yticklabels([f"C{c}" for c in uniq])
    ax2.axvline(gap_mag.mean(), color="k", ls="--", lw=1, label=f"overall mean={gap_mag.mean():.2f}")
    ax2.set_xlabel("mean img−txt gap"); ax2.set_title("Cluster mean cross-modal gap")
    ax2.legend(); ax2.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    print("Each cluster's dominant type and gap reveal what the unsupervised group corresponds to.")
else:
    print("HDBSCAN found <2 clusters — skip cluster profiling (see KMeans best_k view in A2).")

In [ ]:
# ── C3: What do the top principal axes track? ────────────────────────────────
from scipy.stats import pearsonr, f_oneway
scores = PCA(3, random_state=SEED).fit_transform(cond_trained)   # [N, 3]
print("Top-3 condition axes — correlation with candidate factors:")
print(f"{'axis':>5} | {'r(gap)':>8} | {'eta2(type)':>11} | dominant type")
for a in range(3):
    r_gap = pearsonr(scores[:, a], gap_mag)[0]
    grand = scores[:, a].mean()
    ss_between = sum(len(scores[types==t,a]) * (scores[types==t,a].mean()-grand)**2 for t in range(4))
    eta2 = ss_between / ((scores[:, a]-grand)**2).sum()
    dom = TYPE_NAMES[int(np.argmax([scores[types==t,a].mean() for t in range(4)]))]
    print(f"{a:>5} | {r_gap:>8.3f} | {eta2:>11.3f} | high on '{dom}'")
print("→ A high |r(gap)| or eta2(type) names what an axis encodes; near-zero everywhere = uninterpreted axis.")

---
## Summary

In [ ]:
print("=" * 64)
print("  CONDITION STRUCTURE — VERDICT")
print("=" * 64)
def flag(cond, yes, no): return ("✓ " + yes) if cond else ("✗ " + no)

print(f"\nA1 Dimensionality:   PR trained={PR_t:.2f}/{D}")
print("   " + flag(PR_t > 0.4*D, "field uses many dims", "collapsed to low-dim subspace"))
print(f"\nA2 Clusters:         {n_clusters} HDBSCAN, best KMeans k={best_k}, AMI(vs type)={ami:.3f}")
print("   " + flag(best_k != 4 or ami < 0.5, "structure is NOT just the 4 types (bigger picture exists)",
                   "structure ≈ the 4 query types"))
print(f"\nA3 vs null:          silhouette real={sil_real:.3f} vs null={sil_null:.3f}")
print("   " + flag(sil_real > sil_null + 0.02, "genuine clustered structure", "indistinguishable from Gaussian blob"))
print(f"\nB2 Buddy retention:  trained-vs-E={overlaps['trained vs E']:.3f} (init-vs-E={overlaps['init  vs E']:.3f})")
print("   " + flag(overlaps['trained vs E'] > 0.5*overlaps['init  vs E'], "buddy structure retained through training",
                   "training washed out buddy structure"))
side = "image" if overlaps['trained vs A_img'] > overlaps['trained vs A_txt'] else "text"
print(f"   drift toward: {side} (A_img={overlaps['trained vs A_img']:.3f}, A_txt={overlaps['trained vs A_txt']:.3f}); combine_side={combine_side}")
print(f"\nC1 Grounding:        R² img={r2['img features']:.3f} txt={r2['txt features']:.3f} type={r2['query type']:.3f}")
print("   " + flag(max(r2['img features'], r2['txt features']) > 0.2, "conditions grounded in content (learnable)",
                   "conditions look idiosyncratic / memorized"))
print("=" * 64)